# Imports

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models, applications, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve

# Constants

In [2]:
CSV_PATH = "/kaggle/input/competitions/isic-2024-challenge/train-metadata.csv"
IMAGE_DIR = "/kaggle/input/competitions/isic-2024-challenge/train-image/image"

MODEL_SAVE_PATH = "/kaggle/working/skin_cancer_model.keras"

IMG_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 15

# Load metadata

In [3]:
meta_data = pd.read_csv(CSV_PATH, low_memory=False)

meta_data["image_path"] = meta_data["isic_id"].apply(
    lambda x: os.path.join(IMAGE_DIR, f"{x}.jpg")
)

print(meta_data["target"].value_counts())

target
0    400666
1       393
Name: count, dtype: int64


# Train / Validation split

In [4]:
train_df, val_df = train_test_split(
    meta_data,
    test_size=0.2,
    stratify=meta_data["target"],
    random_state=42
)

print(train_df["target"].value_counts())
print(val_df["target"].value_counts())

target
0    320533
1       314
Name: count, dtype: int64
target
0    80133
1       79
Name: count, dtype: int64


# Oversample positives

In [5]:
positive = train_df[train_df.target == 1]
negative = train_df[train_df.target == 0]

positive = positive.sample(
    n=len(negative) // 4,
    replace=True,
    random_state=42
)

train_df = pd.concat([negative, positive])

train_df = (
    train_df
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(train_df.target.value_counts())

target
0    320533
1     80133
Name: count, dtype: int64


# Data augmentation

In [6]:
augment = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1),

    layers.RandomContrast(0.1)

])

I0000 00:00:1783855492.228650      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


# Dataset loader

In [7]:
def process_path(path, label, training=False):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.

    if training:
        image = augment(image)

    return image, label

# tf.data pipelines

In [8]:
train_ds = tf.data.Dataset.from_tensor_slices(

    (
        train_df.image_path.values,
        train_df.target.values
    )

)

train_ds = train_ds.map(
    lambda x,y: process_path(x,y,True),
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = (
    train_ds
    .shuffle(5000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices(

    (
        val_df.image_path.values,
        val_df.target.values
    )

)

val_ds = val_ds.map(
    process_path,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Build model

In [9]:
base_model = applications.EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")

])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# Compile

In [10]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(1e-4),

    loss="binary_crossentropy",

    metrics=[

        tf.keras.metrics.AUC(name="auc"),

        tf.keras.metrics.Precision(),

        tf.keras.metrics.Recall()

    ]

)

# Callbacks

In [11]:
early_stop = callbacks.EarlyStopping(

    monitor="val_auc",

    mode="max",

    patience=5,

    restore_best_weights=True

)

reduce_lr = callbacks.ReduceLROnPlateau(

    monitor="val_auc",

    mode="max",

    factor=0.5,

    patience=2

)

# Train

In [12]:
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=[

        early_stop,

        reduce_lr

    ]

)

Epoch 1/15


2026-07-12 11:25:50.527970: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 11:25:50.733920: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1783855560.046942      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12519/12521 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - auc: 0.5134 - loss: 0.5041 - precision: 0.3446 - recall: 0.0018

2026-07-12 12:09:04.333890: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 12:09:04.542041: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


12521/12521 ━━━━━━━━━━━━━━━━━━━━ 2850s 223ms/step - auc: 0.5363 - loss: 0.4989 - precision: 0.6626 - recall: 0.0062 - val_auc: 0.7664 - val_loss: 0.0451 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 2/15
12521/12521 ━━━━━━━━━━━━━━━━━━━━ 2212s 175ms/step - auc: 0.6492 - loss: 0.4700 - precision: 0.6716 - recall: 0.0793 - val_auc: 0.8112 - val_loss: 0.1654 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 3/15
12521/12521 ━━━━━━━━━━━━━━━━━━━━ 2165s 171ms/step - auc: 0.6830 - loss: 0.4584 - precision: 0.6718 - recall: 0.1149 - val_auc: 0.8270 - val_loss: 0.0766 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 4/15
12521/12521 ━━━━━━━━━━━━━━━━━━━━ 2168s 171ms/step - auc: 0.7007 - loss: 0.4507 - precision: 0.6800 - recall: 0.1371 - val_auc: 0.8250 - val_loss: 0.1227 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 5/15
12521/12521 ━━━━━━━━

# Save model

In [13]:
model.save(MODEL_SAVE_PATH)

# Predict on validation set

In [14]:
predictions = model.predict(val_ds).ravel()

y_true = val_df.target.values

2507/2507 ━━━━━━━━━━━━━━━━━━━━ 76s 28ms/step


# Find best threshold

In [15]:
precision, recall, thresholds = precision_recall_curve(

    y_true,

    predictions

)

f1 = 2 * precision * recall / (precision + recall + 1e-8)

best = np.argmax(f1)

print("Best Threshold :", thresholds[best])
print("F1 :", f1[best])
print("Precision :", precision[best])
print("Recall :", recall[best])

Best Threshold : 0.6685166
F1 : 0.07216494362578416
Precision : 0.06086956521739131
Recall : 0.08860759493670886
